# Long-term memory without duplicates: one profile per user

`in_memory_store.ipynb` saved every fact as a **new item**, so repeating yourself made duplicates and
"I moved to Bangalore" left "from Jaipur" behind. Here each user has **one profile** stored under one key.
The LLM gets the current profile + the new message and returns the **whole updated profile**:
merged, de-duplicated, outdated facts replaced. Saving overwrites the old profile.

```
START -> remember (read profile, update it, save it) -> chat (answer using the profile) -> END
```

In [1]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(model='openai/gpt-oss-20b', reasoning_effort='low')

In [2]:
class UserProfile(BaseModel):
    changed: bool = Field(description='True if the new message added or changed anything in the profile')
    facts: list[str] = Field(description='The COMPLETE updated list of facts about the user, each written as "User ..."')


profile_updater = model.with_structured_output(UserProfile, method='json_schema')

PROFILE_KEY = 'profile'  # one key per user -> one profile per user, so duplicates can't pile up


def get_profile(store, user_id):
    item = store.get((user_id, 'memories'), PROFILE_KEY)
    return item.value['facts'] if item else []

In [3]:
# Node 1: remember. Reads the profile, lets the LLM update it, overwrites it in the store.
def remember(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    user_id = config['configurable']['user_id']
    current = get_profile(store, user_id)

    result = profile_updater.invoke(
        f'Current profile of the user: {current}\n'
        f"New user message: {state['messages'][-1].content}\n\n"
        'Return the complete updated profile:\n'
        '- keep every existing fact that is still true\n'
        '- add new lasting facts (name, city, job, likes, goals), one per item, written as "User ..."\n'
        '- if a fact changed (e.g. the user moved), REPLACE the old fact instead of keeping both\n'
        '- never list the same fact twice, even in different words\n'
        '- questions and small talk change nothing: return the profile as it is with changed=false'
    )
    if result.changed:
        store.put((user_id, 'memories'), PROFILE_KEY, {'facts': result.facts})  # put on the same key = overwrite
    return {}


# Node 2: chat. Answers using the profile plus this thread's messages.
def chat(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    facts = get_profile(store, config['configurable']['user_id'])
    known = '\n'.join(f'- {fact}' for fact in facts) or '(nothing yet)'
    system = SystemMessage(f'You are a helpful assistant. What you know about this user:\n{known}')
    return {'messages': [model.invoke([system] + state['messages'])]}

In [4]:
graph = StateGraph(MessagesState)
graph.add_node('remember', remember)
graph.add_node('chat', chat)
graph.add_edge(START, 'remember')
graph.add_edge('remember', 'chat')
graph.add_edge('chat', END)

store = InMemoryStore()
chatbot = graph.compile(checkpointer=InMemorySaver(), store=store)


def talk(text, user_id, thread_id):
    config = {'configurable': {'user_id': user_id, 'thread_id': thread_id}}
    result = chatbot.invoke({'messages': [HumanMessage(content=text)]}, config=config)
    print(f'[{thread_id}] You: {text}\nAI: {result["messages"][-1].content[:140]}\n')


def show_profile(user_id):
    facts = get_profile(store, user_id)
    print(f'{user_id} profile ({len(facts)} facts):')
    for fact in facts:
        print('  -', fact)
    print()

In [5]:
talk('Hi! I am Priyansh from Jaipur. I work as a data analyst.', 'user-1', 'chat-1')
talk('I love cricket and I am learning LangGraph.', 'user-1', 'chat-1')
show_profile('user-1')

[chat-1] You: Hi! I am Priyansh from Jaipur. I work as a data analyst.
AI: Hello Priyansh! 👋  
Nice to meet you—welcome to the chat! How can I help you today? Whether it’s data analysis tips, insights on tools, or a



[chat-1] You: I love cricket and I am learning LangGraph.
AI: That’s awesome! 🏏

- **Cricket** – Do you have a favorite team or player?  
- **LangGraph** – Are you working on a specific project, or just

user-1 profile (5 facts):
  - User name is Priyansh
  - User city is Jaipur
  - User job is data analyst
  - User likes cricket
  - User goal is learning LangGraph



In [6]:
# Repeat the same facts in other words, in new threads: the profile must NOT grow
talk('Hey, Priyansh here. Big cricket fan, based in Jaipur.', 'user-1', 'chat-2')
talk('As a data analyst who is studying LangGraph, what should I build?', 'user-1', 'chat-3')
show_profile('user-1')

[chat-2] You: Hey, Priyansh here. Big cricket fan, based in Jaipur.
AI: Hey Priyansh! 👋  
Nice to meet a fellow cricket enthusiast from Jaipur. How’s the weather treating you today?  

You mentioned you’re lookin



[chat-3] You: As a data analyst who is studying LangGraph, what should I build?
AI: Hey Priyansh!  
As a data analyst in Jaipur who’s diving into LangGraph, you’re in a sweet spot to build something that blends real‑world da

user-1 profile (5 facts):
  - User name is Priyansh
  - User city is Jaipur
  - User job is data analyst
  - User likes cricket
  - User goal is learning LangGraph



In [7]:
# A changed fact: Jaipur must be REPLACED by Bangalore, not kept alongside it
talk('I have moved to Bangalore now.', 'user-1', 'chat-4')
show_profile('user-1')

# New thread, so no chat history: the answer can only come from long-term memory
talk('Where do I live and what do I do?', 'user-1', 'chat-5')

[chat-4] You: I have moved to Bangalore now.
AI: Hey Priyansh! 🚀

Welcome to Bangalore – the city that’s as vibrant and dynamic as your data analytics career. I’m sure you’ll find plenty of

user-1 profile (5 facts):
  - User name is Priyansh
  - User city is Bangalore
  - User job is data analyst
  - User likes cricket
  - User goal is learning LangGraph



[chat-5] You: Where do I live and what do I do?
AI: You live in Bangalore, you work as a data analyst, you enjoy cricket, and your current goal is to learn LangGraph.

